In [ ]:
# Colab reproducibility — uncomment to clone the repo and enter this folder:
# !git clone https://github.com/JuliettKhar/mlops-zoomcamp-exercises.git
# %cd mlops-zoomcamp-exercises/04-deploy/web-service

In [35]:
!pip freeze | grep scikit-learn

scikit-learn @ file:///Users/runner/miniforge3/conda-bld/bld/rattler-build_scikit-learn_1766550894/work/dist/scikit_learn-1.8.0-cp313-cp313-macosx_11_0_arm64.whl#sha256=25b313b21327b8ed13e5a87685b7dc359e267890bdaa017d5aefc838b32f89c8


In [36]:
!python -V

Python 3.13.12


In [69]:
year = 2026
month = 3

In [37]:
import pickle
import pandas as pd
from sklearn.metrics import root_mean_squared_error

In [38]:
with open('models/lin_reg.bin', 'rb') as f_in:
    dv, model = pickle.load(f_in)

In [ ]:
categorical = ['PU_DO']

def read_data(filename):
    df = pd.read_parquet(filename)
    
    df['duration'] = df.tpep_dropoff_datetime - df.tpep_pickup_datetime
    df['duration'] = df.duration.dt.total_seconds() / 60

    df['PULocationID'] = df['PULocationID'].fillna(-1).astype('int').astype('str')
    df['DOLocationID'] = df['DOLocationID'].fillna(-1).astype('int').astype('str')

    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']
    
    return df

In [70]:
df = read_data(f'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{year}-{month:02d}.parquet')

In [73]:
dicts = df[categorical].to_dict(orient='records')
X_val = dv.transform(dicts)
print(X_val.nnz)
y_train = df.duration.values
y_pred = model.predict(X_val)

922831


In [75]:

rmse = root_mean_squared_error(y_train, y_pred)

In [76]:
print("Rows:", len(df))
print("Mean:", y_pred.mean())
print("Std:", y_pred.std())
print("Min:", y_pred.min())
print("Max:", y_pred.max())

Rows: 3952451
Mean: 24.420650031646737
Std: 4.8415537166399485
Min: 0.271054353842235
Max: 86.7764908514376


In [77]:
print(type(model))
print(model)
print(X_val.shape)
print(X_val.nnz)

<class 'sklearn.linear_model._ridge.Ridge'>
Ridge()
(3952451, 13559)
922831


In [90]:
print(f'Q1. Notebook: {y_pred.std()}')

Q1. Notebook: 4.8415537166399485


In [82]:
df['ride_id'] = f'{year:04d}/{month:02d}_' + df.index.astype('str')
df['ride_id'] = f'{year:04d}/{month:02d}_' + df.index.astype('str')

df_res = pd.DataFrame()
df_res['ride_id'] = df['ride_id']
df_res['predicted_duration'] = y_pred

In [83]:
df_res.head()
df_res.shape

(3952451, 2)

In [84]:
output_file = f'yellow_tripdata_{year}-{month}_predictions.parquet'
df_res.to_parquet(
    output_file,
    engine='pyarrow',
    compression=None,
    index=False
)


In [86]:
df_res.head()

,ride_id,predicted_duration
0,2026/03_0,26.196409
1,2026/03_1,26.196409
2,2026/03_2,26.196409
3,2026/03_3,21.493351
4,2026/03_4,26.196409


In [88]:
!ls -lh yellow_tripdata_2026-3_predictions.parquet

-rw-r--r--@ 1 julia  staff    76M Jun  2 09:17 yellow_tripdata_2026-3_predictions.parquet


In [89]:
import os

size_mb = os.path.getsize(output_file) / 1024 / 1024
print(f'Q2. Preparing the output: {size_mb}')

Q2. Preparing the output: 76.44007587432861


In [ ]:
# Q3. Creating the scoring script
# jupyter nbconvert --to=script starter.ipynb     

In [92]:
!pip freeze | grep scikit-learn

scikit-learn @ file:///Users/runner/miniforge3/conda-bld/bld/rattler-build_scikit-learn_1766550894/work/dist/scikit_learn-1.8.0-cp313-cp313-macosx_11_0_arm64.whl#sha256=25b313b21327b8ed13e5a87685b7dc359e267890bdaa017d5aefc838b32f89c8


In [96]:
# Q4. Virtual Environment

In [94]:
# sha256:00d6f1d66fbcf4eba6e356e1420d33cc06c70a45bb1363cd6f6a8e4ebbbdece2